# Chapter 2: Routing

EliteAI Academy

Route requests with rules, LLMs, embeddings, and classifiers; combine nested routing with prompt chains.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
        r = _client.messages.create(thinking={"type": "disabled"}, 
            model=_model, max_tokens=512, system=system,
            messages=[{"role": "user", "content": prompt}]
        )
        return r.content[0].text.strip()

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
        full_prompt = f"{system}\n\n{prompt}"
        return _gmodel.generate_content(full_prompt).text.strip()

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply with exactly: ROUTING READY"))

## 1. The Problem — Why Sequential Chains Break Down

A prompt chain is linear: A → B → C. But real requests need **branching**: *if booking → BookingAgent, if info → InfoAgent, else → Clarify.*

Let's demonstrate the failure of a naive chain on diverse inputs.

In [ ]:
def naive_handler(request: str) -> str:
    return call_llm(
        f"Handle this user request as best you can: {request}",
        system="You are a generic assistant. Handle all requests the same way."
    )

test_requests = [
    "Book me a flight to Lagos for next Friday.",
    "What is the boiling point of water?",
    "Asdfjkl?",
    "My order #4521 hasn't arrived yet.",
]

print("=== NAIVE SINGLE-CHAIN RESPONSES ===")
for req in test_requests:
    print(f"\n📨 Request: {req}")
    print(f"🤖 Response: {naive_handler(req)}")
    print("-" * 55)

print("\n⚠️  All requests get the same generic treatment.")
print("   No specialised handling. No escalation. No tools.")

## 2. Routing Mechanism Type 1 — Rule-Based Routing

Fastest and most deterministic. Uses keyword/pattern matching — no LLM call needed.

In [ ]:
import re

ROUTING_RULES = {
    "booking":  ["book", "flight", "hotel", "reserve", "reservation", "ticket"],
    "order":    ["order", "delivery", "shipment", "track", "arrived", "package"],
    "info":     ["what", "how", "explain", "tell me", "define", "capital", "boiling"],
    "support":  ["broken", "error", "bug", "not working", "issue", "problem", "crash"],
}

def rule_based_router(request: str) -> str:
    'Route based on keyword matching. Returns route name.'
    text = request.lower()
    for route, keywords in ROUTING_RULES.items():
        if any(kw in text for kw in keywords):
            return route
    return "unclear"

def booking_handler(req):  return f"[BOOKING AGENT]  Processing: '{req}' → Checking availability..."
def order_handler(req):    return f"[ORDER AGENT]    Looking up order for: '{req}'"
def info_handler(req):     return f"[INFO AGENT]     Fetching knowledge base for: '{req}'"
def support_handler(req):  return f"[SUPPORT AGENT]  Opening support ticket for: '{req}'"
def unclear_handler(req):  return f"[CLARIFIER]      Could not route: '{req}'. Please rephrase."

HANDLER_MAP = {
    "booking":  booking_handler,
    "order":    order_handler,
    "info":     info_handler,
    "support":  support_handler,
    "unclear":  unclear_handler,
}

def dispatch(request: str, router_fn) -> str:
    route = router_fn(request)
    handler = HANDLER_MAP[route]
    return route, handler(request)

print("=== RULE-BASED ROUTING ===")
for req in test_requests:
    route, result = dispatch(req, rule_based_router)
    print(f"\n📨 Request : {req}")
    print(f"🔀 Route   : {route.upper()}")
    print(f"✅ Result  : {result}")

### Observation

Rules are fast but can miss nuanced requests.

## 3. Routing Mechanism Type 2 — LLM-Based Routing

An LLM classifies the request, including implicit intent.

In [ ]:
ROUTER_SYSTEM = """You are an intent classifier for a customer service system.
Analyse the user's request and output ONLY one word — the route:
- 'booking'  → flights, hotels, reservations
- 'order'    → order status, delivery, shipment tracking  
- 'info'     → general knowledge, factual questions
- 'support'  → bugs, errors, technical issues
- 'unclear'  → anything ambiguous or none of the above

Output ONLY the single route word. Nothing else."""

def llm_router(request: str) -> str:
    'Route using LLM intent classification.'
    raw = call_llm(request, system=ROUTER_SYSTEM).lower().strip()

    route = raw.split()[0] if raw.split() else "unclear"
    return route if route in HANDLER_MAP else "unclear"

print("=== LLM-BASED ROUTING ===")
nuanced_requests = [
    "I need to get to Paris for a conference next Tuesday.",
    "My parcel seems to have gone missing.",
    "The app keeps crashing whenever I tap the settings icon.",
    "Who invented the telephone?",
    "Blah blah blah.",
]

for req in nuanced_requests:
    route, result = dispatch(req, llm_router)
    print(f"\n📨 Request : {req}")
    print(f"🔀 Route   : {route.upper()}")
    print(f"✅ Result  : {result}")

### Observation

LLM routing adds latency and API cost.

## 4. Routing Mechanism Type 3 — Embedding-Based Semantic Routing

Compare request embeddings with route prototypes using cosine similarity.

In [ ]:
import numpy as np

ROUTE_PROTOTYPES = {
    "booking":  "book flight hotel reservation ticket travel accommodation",
    "order":    "order delivery shipment tracking package arrived dispatch",
    "info":     "what how explain define capital history science knowledge",
    "support":  "broken error bug crash problem issue not working technical",
    "unclear":  "random unclear ambiguous nonsense gibberish unrelated",
}

def simple_embed(text: str, vocab: list) -> np.ndarray:
    'Bag-of-words embedding over a shared vocabulary.'
    words = set(text.lower().split())
    return np.array([1.0 if w in words else 0.0 for w in vocab])

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

all_words = " ".join(ROUTE_PROTOTYPES.values()).split()
VOCAB = sorted(set(all_words))

prototype_embeddings = {
    route: simple_embed(desc, VOCAB)
    for route, desc in ROUTE_PROTOTYPES.items()
}

def embedding_router(request: str) -> str:
    'Route by cosine similarity between request and route prototypes.'
    req_vec = simple_embed(request, VOCAB)
    scores = {
        route: cosine_similarity(req_vec, proto_vec)
        for route, proto_vec in prototype_embeddings.items()
    }
    best_route = max(scores, key=scores.get)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print(f"  Similarity scores: {[(r, round(s,3)) for r,s in sorted_scores]}")
    return best_route if scores[best_route] > 0.0 else "unclear"

print("=== EMBEDDING-BASED SEMANTIC ROUTING ===")
embed_test = [
    "I want to reserve a room in a hotel in Dubai.",
    "Where is my package?",
    "Explain photosynthesis.",
    "The software is not working properly.",
]
for req in embed_test:
    print(f"\n📨 Request : {req}")
    route, result = dispatch(req, embedding_router)
    print(f"🔀 Route   : {route.upper()}")
    print(f"✅ Result  : {result}")

Use a real embedding model to replace `simple_embed`.

## 5. Routing Type 4 — ML Classifier-Based Routing

Train a classifier on labelled examples, then route locally.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

TRAINING_DATA = [

    ("Book a flight to London", "booking"),
    ("Reserve a hotel room in Paris", "booking"),
    ("I need to book two tickets to New York", "booking"),
    ("Can you find me accommodation in Tokyo?", "booking"),
    ("I want to reserve seats on the 7pm train", "booking"),

    ("Where is my order?", "order"),
    ("Track my shipment #4521", "order"),
    ("My delivery hasn't arrived", "order"),
    ("When will my package be delivered?", "order"),
    ("I want to check the status of my parcel", "order"),

    ("What is the capital of France?", "info"),
    ("Explain quantum entanglement", "info"),
    ("How does photosynthesis work?", "info"),
    ("Who invented electricity?", "info"),
    ("Tell me about the French Revolution", "info"),

    ("The app keeps crashing", "support"),
    ("I'm getting an error on login", "support"),
    ("The feature is not working", "support"),
    ("There's a bug in the checkout page", "support"),
    ("My account is broken", "support"),
]

texts, labels = zip(*TRAINING_DATA)

clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("lr",    LogisticRegression(max_iter=1000, C=5))
])
clf.fit(texts, labels)
print("✅ Classifier trained on", len(texts), "examples")

def ml_router(request: str) -> str:
    'Route using trained ML classifier.'
    pred = clf.predict([request])[0]
    proba = clf.predict_proba([request])[0]
    classes = clf.classes_
    scores = dict(zip(classes, [round(p, 3) for p in proba]))
    print(f"  Confidence scores: {scores}")
    return pred

print("\n=== ML CLASSIFIER-BASED ROUTING ===")
ml_test = [
    "Find me a room near the airport.",
    "My parcel is three weeks late.",
    "Why is the sky blue?",
    "The dashboard shows a 500 error.",
]
for req in ml_test:
    print(f"\n📨 Request : {req}")
    route, result = dispatch(req, ml_router)
    print(f"🔀 Route   : {route.upper()}")
    print(f"✅ Result  : {result}")

## 6. Routing Mechanism Comparison

Quick reference summary across all four types.

In [ ]:
comparison = """
╔══════════════════╦══════════╦═══════════╦═════════════╦══════════════════════╗
║ Mechanism        ║ Speed    ║ Accuracy  ║ API Cost    ║ Best For             ║
╠══════════════════╬══════════╬═══════════╬═════════════╬══════════════════════╣
║ Rule-Based       ║ ⚡⚡⚡   ║ ⭐⭐       ║ Free        ║ Fixed, well-defined  ║
║                  ║          ║           ║             ║ high-volume routing  ║
╠══════════════════╬══════════╬═══════════╬═════════════╬══════════════════════╣
║ LLM-Based        ║ ⚡        ║ ⭐⭐⭐⭐⭐  ║ Per-call    ║ Nuanced / novel      ║
║                  ║          ║           ║             ║ inputs, complex NLU  ║
╠══════════════════╬══════════╬═══════════╬═════════════╬══════════════════════╣
║ Embedding-Based  ║ ⚡⚡⚡   ║ ⭐⭐⭐⭐   ║ Embed only  ║ Semantic similarity, ║
║                  ║          ║           ║             ║ multilingual inputs  ║
╠══════════════════╬══════════╬═══════════╬═════════════╬══════════════════════╣
║ ML Classifier    ║ ⚡⚡⚡   ║ ⭐⭐⭐⭐   ║ Free after  ║ Stable, labelled     ║
║                  ║          ║           ║ training    ║ domains, offline use ║
╚══════════════════╩══════════╩═══════════╩═════════════╩══════════════════════╝
"""
print(comparison)

## 7. LangChain LCEL Implementation — `RunnableBranch`

The book's code example: a coordinator using `RunnableBranch` to delegate to handlers.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch

lc_llm = ChatAnthropic(model_kwargs={"thinking": {"type": "disabled"}}, model=MODEL, temperature=0)

router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the user's request and classify it.
- If related to booking flights or hotels, output exactly: booker
- For general information questions, output exactly: info
- If unclear or doesn't fit, output exactly: unclear
Output ONLY one word."""),
    ("user", "{request}")
])

def lc_booking_handler(x):  return f"[BOOKING AGENT] Processing: '{x['request']['request']}'"
def lc_info_handler(x):     return f"[INFO AGENT]    Answering: '{x['request']['request']}'"
def lc_unclear_handler(x):  return f"[CLARIFIER]     Could not route: '{x['request']['request']}'. Please rephrase."

router_chain = router_prompt | lc_llm | StrOutputParser()

delegation_branch = RunnableBranch(
    (lambda x: x["decision"].strip() == "booker",  RunnablePassthrough.assign(output=lc_booking_handler)),
    (lambda x: x["decision"].strip() == "info",    RunnablePassthrough.assign(output=lc_info_handler)),
    RunnablePassthrough.assign(output=lc_unclear_handler)
)

coordinator = (
    {"decision": router_chain, "request": RunnablePassthrough()}
    | delegation_branch
    | (lambda x: {"route": x["decision"].strip(), "result": x["output"]})
)

lc_requests = [
    "Book me a flight to London.",
    "What is the capital of Italy?",
    "Tell me about quantum physics.",
]

print("=== LangChain RunnableBranch Router ===")
for req in lc_requests:
    out = coordinator.invoke({"request": req})
    print(f"\n📨 Request : {req}")
    print(f"🔀 Route   : {out['route'].upper()}")
    print(f"✅ Result  : {out['result']}")

## 8. Multi-Level / Nested Routing

Real systems have **hierarchical routing**: a top-level router dispatches to domains, then a second router within each domain picks the right sub-action.

In [ ]:
L1_SYSTEM = """Classify this request into one of these domains.
Output ONLY one word: 'travel', 'finance', 'tech', or 'other'."""

L2_SYSTEMS = {
    "travel":  """Classify: 'flight', 'hotel', 'car_rental', or 'other'. Output one word.""",
    "finance": """Classify: 'account', 'payment', 'investment', or 'other'. Output one word.""",
    "tech":    """Classify: 'bug_report', 'feature_request', 'how_to', or 'other'. Output one word.""",
}

def nested_router(request: str) -> dict:
    'Two-level hierarchical router.'

    domain = call_llm(request, system=L1_SYSTEM).lower().split()[0]
    if domain not in L2_SYSTEMS:
        return {"domain": "other", "sub_route": "other", "handler": "general"}

    sub_route = call_llm(request, system=L2_SYSTEMS[domain]).lower().split()[0]
    handler = f"{domain}::{sub_route}"
    return {"domain": domain, "sub_route": sub_route, "handler": handler}

nested_tests = [
    "I want to rent a car in Abuja next week.",
    "How do I rebalance my investment portfolio?",
    "How do I reset my API key in the developer portal?",
    "Can you add dark mode to the mobile app?",
]

print("=== NESTED (TWO-LEVEL) ROUTING ===")
for req in nested_tests:
    routing = nested_router(req)
    print(f"\n📨 Request  : {req}")
    print(f"🔀 L1 Domain: {routing['domain'].upper()}")
    print(f"🔀 L2 Sub   : {routing['sub_route'].upper()}")
    print(f"📬 Handler  : {routing['handler']}")

## 9. Full End-to-End Example — AI Coding Assistant Router

Identifies: (1) programming language and (2) user intent (debug / explain / translate), then routes to a specialised code handler.

In [ ]:
CODE_ROUTER_SYSTEM = """You are a coding assistant router.
Analyze the request and output ONLY a JSON object with two keys:
- "language": the programming language (e.g. python, javascript, sql, unknown)
- "intent": one of debug, explain, translate, write, other

Example output: {"language": "python", "intent": "debug"}
Return ONLY the JSON. No markdown. No explanation."""

def code_router(request: str) -> dict:
    raw = call_llm(request, system=CODE_ROUTER_SYSTEM)

    clean = re.sub(r"```[\w]*", "", raw).strip()
    return json.loads(clean)

def code_debug_handler(req, lang):    return call_llm(f"Debug this {lang} code and explain the fix:\n{req}")
def code_explain_handler(req, lang):  return call_llm(f"Explain this {lang} code clearly for a junior developer:\n{req}")
def code_translate_handler(req, lang):return call_llm(f"Translate this {lang} code to Python. Keep the same logic:\n{req}")
def code_write_handler(req, lang):    return call_llm(f"Write clean {lang} code for this requirement:\n{req}")
def code_other_handler(req, lang):    return call_llm(f"Help with this coding question:\n{req}")

INTENT_HANDLERS = {
    "debug":     code_debug_handler,
    "explain":   code_explain_handler,
    "translate": code_translate_handler,
    "write":     code_write_handler,
}

def coding_assistant(request: str) -> str:
    routing = code_router(request)
    lang   = routing.get("language", "unknown")
    intent = routing.get("intent", "other")
    print(f"  🔀 Detected: language={lang.upper()}, intent={intent.upper()}")
    handler = INTENT_HANDLERS.get(intent, code_other_handler)
    return handler(request, lang)

code_requests = [
    """Explain what this does:\nfor i in range(len(lst)-1, -1, -1):\n    lst.pop(i)""",
    """Debug this: SELECT * FROM orders WHERE customer_id = NULL""",
    "Write a function that checks if a string is a palindrome.",
]

print("=== AI CODING ASSISTANT ROUTER ===")
for req in code_requests:
    print(f"\n📨 Request:\n{req}")
    result = coding_assistant(req)
    print(f"\n🤖 Response:\n{result}")
    print("=" * 60)

## 10. Routing + Chaining Combined

Route first → then execute a full prompt chain within the chosen branch. This is the foundation of multi-agent orchestration.

In [ ]:
def booking_chain(request: str) -> str:
    'A 3-step chain that runs ONLY for booking requests.'

    details_raw = call_llm(
        f"Extract trip details from this request as JSON with keys: destination, dates, type (flight/hotel/both).\nRequest: {request}\nReturn ONLY JSON."
    )
    details = json.loads(re.sub(r"```[\w]*", "", details_raw).strip())
    print(f"  [Chain Step 1] Extracted details: {details}")

    availability = f"3 options available for {details.get('type','flight')} to {details.get('destination','destination')} on {details.get('dates','your dates')}."
    print(f"  [Chain Step 2] Availability: {availability}")

    summary = call_llm(
        f"Write a friendly 2-sentence booking summary for the user based on:\nRequest: {request}\nAvailability: {availability}"
    )
    print(f"  [Chain Step 3] Summary generated")
    return summary

def info_chain(request: str) -> str:
    'A 2-step chain for information requests.'

    core_q = call_llm(f"Restate the core question in one sentence: {request}")
    print(f"  [Chain Step 1] Core question: {core_q}")

    answer = call_llm(f"Answer this question in 3 sentences: {core_q}")
    return answer

CHAINED_HANDLERS = {"booking": booking_chain, "info": info_chain}

def router_plus_chain(request: str) -> str:
    route = llm_router(request)
    print(f"\n🔀 Routed to: {route.upper()}")
    handler = CHAINED_HANDLERS.get(route, lambda r: call_llm(r))
    return handler(request)

print("=== ROUTING + CHAINING COMBINED ===")
combined_tests = [
    "I need to book a flight to Abuja this Saturday.",
    "How does the TCP/IP protocol work?",
]
for req in combined_tests:
    print(f"\n📨 Request: {req}")
    result = router_plus_chain(req)
    print(f"\n✅ Final Result:\n{result}")
    print("=" * 60)

## 11. Summary — Pattern Rules of Thumb

In [ ]:
summary = """
╔══════════════════════════════════╦═════════════════════════════════════════════╗
║ When to USE Routing              ║ Key Design Rules                            ║
╠══════════════════════════════════╬═════════════════════════════════════════════╣
║ Agent handles diverse request    ║ Keep router prompt focused — one output     ║
║ types that need different tools  ║ word / category only                        ║
║ Need to triage/classify inputs   ║ Sanitise router output before branching     ║
║ Multiple specialist agents exist ║ Always have a default / fallback route      ║
║ Sequential chain is too rigid    ║ Combine routing + chaining for each branch  ║
║ Context determines best action   ║ Use rule-based first; LLM for edge cases    ║
╚══════════════════════════════════╩═════════════════════════════════════════════╝
"""
print(summary)

## 12. Exercises

- **Exercise 1 — Hybrid Router:** Build a router that uses rule-based matching first (for speed), and falls back to LLM classification only when no keyword match is found.

- **Exercise 2 — Confidence Threshold:** Modify `ml_router` to fall back to the `unclear` handler when the top class probability is below 0.6.

- **Exercise 3 — Three-Level Nesting:** Extend the nested router to a third level. Example: `finance → investment → buy/sell/hold`.

- **Exercise 4 — Real Embeddings:** Replace `simple_embed` in Section 4 with `sentence-transformers` (`pip install sentence-transformers`). Use the `all-MiniLM-L6-v2` model. Does routing accuracy improve?

- **Exercise 5 — LangGraph Router:** Refactor the LangChain `RunnableBranch` example (Section 7) into a LangGraph `StateGraph` with typed nodes and conditional edges.

**Next:** Chapter 3 — Parallelisation & Fan-Out Patterns